In [ ]:
import math
import time
import numpy as np
import networkx as nx
from joblib import Parallel, delayed

# =============================================================================
# OPERAD SIGNATURE BITPACKING LAYER (PosetOperad)
# =============================================================================
class PosetOperad:
    """
    High-performance, memory-optimized computing pipeline for algebraic
    poset operads on finite poset matrices supporting all 9 boundary partition variants.
    """

    @staticmethod
    def matrix_to_bits(matrix: np.ndarray) -> int:
        flat_bin = matrix.ravel()
        weights = 1 << np.arange(flat_bin.size, dtype=object)
        return int(np.sum(flat_bin * weights))

    @staticmethod
    def bits_to_matrix(signature: int, side_dim: int) -> np.ndarray:
        n_elements = side_dim * side_dim
        flat_bin = np.array([(signature >> idx) & 1 for idx in range(n_elements)], dtype='int64')
        return flat_bin.reshape(side_dim, side_dim)

    @classmethod
    def _get_extremal_indices(cls, matrix: np.ndarray) -> tuple[list[int], list[int]]:
        col_sums = matrix.sum(axis=0)
        row_sums = matrix.sum(axis=1)
        max_elements = np.where(col_sums == 1)[0].tolist()
        min_elements = np.where(row_sums == 1)[0].tolist()
        return max_elements, min_elements

    @classmethod
    def compute_operadic_submatrix_V_bitpacked(cls, l_mat, r_mat, i, p, q):
        n, m = p, q
        base_relations_col = l_mat[i:n, i-1]
        v_block = np.zeros((n - i, m), dtype='int64')
        _, min_elements = cls._get_extremal_indices(r_mat)
        if min_elements:
            v_block[:, min_elements] = base_relations_col[:, np.newaxis]
        return v_block

    @classmethod
    def compute_operadic_submatrix_U_bitpacked(cls, l_mat, r_mat, i, p, q):
        row_idx = i - 1
        base_relations = l_mat[row_idx, :row_idx]
        m = q
        u_block = np.zeros((m, row_idx), dtype='int64')
        max_elements, _ = cls._get_extremal_indices(r_mat)
        if max_elements:
            u_block[max_elements] = base_relations
        return u_block

    @classmethod
    def generate_compositions_bitpacked(cls, l_signature: int, r_signature: int,
                                        p: int, q: int, comp_type: str = "general") -> list[int]:
        l_mat = cls.bits_to_matrix(l_signature, p)
        r_mat = cls.bits_to_matrix(r_signature, q)
        output_signatures = []

        for i in range(1, p + 1):
            idx = i - 1
            A11 = l_mat[:idx, :idx]
            A21 = l_mat[i:, i:]

            y1 = np.zeros((idx, q + p - i), dtype='int64')
            y2 = np.zeros((q, p - i), dtype='int64')

            if comp_type == "general":
                A22 = l_mat[i:, :idx]
                row_Ai = l_mat[idx, :idx]
                col_Ai = l_mat[i:, idx]
                Ui = np.outer(np.ones(q, dtype='int64'), row_Ai)
                Vi = np.outer(col_Ai, np.ones(q, dtype='int64'))

            elif comp_type == "minimal":
                A22 = l_mat[i:, :idx]
                row_Ai = l_mat[idx, :idx]
                Ui = np.tile(row_Ai, (q, 1))
                Vi = cls.compute_operadic_submatrix_V_bitpacked(l_mat, r_mat, i, p, q)

            elif comp_type == "maximal":
                A22 = l_mat[i:, :idx]
                Ui = cls.compute_operadic_submatrix_U_bitpacked(l_mat, r_mat, i, p, q)
                col_Ai = l_mat[i:, idx]
                Vi = np.tile(col_Ai[:, np.newaxis], (1, q))

            elif comp_type == "boundary":
                A22 = l_mat[i:, :idx]
                Ui = cls.compute_operadic_submatrix_U_bitpacked(l_mat, r_mat, i, p, q)
                Vi = cls.compute_operadic_submatrix_V_bitpacked(l_mat, r_mat, i, p, q)

            elif comp_type == "total":
                A22 = l_mat[i:, :idx]
                Ui = np.ones((q, idx), dtype='int64')
                Vi = np.ones((p - i, q), dtype='int64')

            elif comp_type == "isolated":
                A22 = np.ones((p - i, idx), dtype='int64')
                Ui = np.zeros((q, idx), dtype='int64')
                Vi = np.zeros((p - i, q), dtype='int64')

            elif comp_type == "pred_dominant":
                A22 = np.ones((p - i, idx), dtype='int64')
                Ui = np.ones((q, idx), dtype='int64')
                Vi = np.zeros((p - i, q), dtype='int64')

            elif comp_type == "succ_dominant":
                A22 = np.ones((p - i, idx), dtype='int64')
                Ui = np.zeros((q, idx), dtype='int64')
                Vi = np.ones((p - i, q), dtype='int64')

            elif comp_type == "fully_isolated":
                A22 = np.zeros((p - i, idx), dtype='int64')
                Ui = np.zeros((q, idx), dtype='int64')
                Vi = np.zeros((p - i, q), dtype='int64')

            full_composed_matrix = np.block([
                [A11, y1],
                [Ui, r_mat, y2],
                [A22, Vi, A21]
            ])
            output_signatures.append(cls.matrix_to_bits(full_composed_matrix))

        return output_signatures


# =============================================================================
# ORDER STRUCTURAL FILTER ENGINE LAYER (PosetProcessor)
# =============================================================================
class PosetProcessor:
    """Unified classification properties workbench for finite partially ordered sets."""

    @classmethod
    def get_maximal_elements(cls, poset_matrix):
        column_sums = poset_matrix.sum(axis=0)
        return np.where(column_sums == 1)[0].tolist()

    @classmethod
    def get_minimal_elements(cls, poset_matrix):
        row_sums = poset_matrix.sum(axis=1)
        return np.where(row_sums == 1)[0].tolist()

    @classmethod
    def compute_dual_poset_matrix(cls, poset_matrix):
        return np.flip(poset_matrix).T

    @classmethod
    def is_not_self_dual(cls, poset_matrix):
        dual = cls.compute_dual_poset_matrix(poset_matrix)
        return not np.array_equal(dual, poset_matrix)

    @classmethod
    def is_trivial_poset(cls, poset_matrix):
        return poset_matrix.shape == (1, 1) and poset_matrix[0, 0] == 1

    @classmethod
    def is_connected_poset(cls, poset_matrix):
        if poset_matrix.size == 0:
            return False
        if poset_matrix.shape[0] == 1:
            return cls.is_trivial_poset(poset_matrix)
        graph = nx.from_numpy_array(poset_matrix)
        return nx.is_connected(graph)

    @classmethod
    def is_semi_right_dualizable(cls, poset_matrix):
        n = poset_matrix.shape[0]
        depth = 0
        while depth < n:
            if np.all(poset_matrix[depth:, depth]):
                depth += 1
            else:
                break
        if depth == 0 or depth == n:
            return False
        submatrix = poset_matrix[depth:, depth:]
        if not cls.is_connected_poset(submatrix):
            is_self_dual = np.array_equal(cls.compute_dual_poset_matrix(submatrix), submatrix)
            if not is_self_dual:
                return True
        return False

    @classmethod
    def is_semi_left_dualizable(cls, poset_matrix):
        n = poset_matrix.shape[0]
        depth = 0
        while depth < n:
            row_idx = n - depth - 1
            if np.all(poset_matrix[row_idx, :row_idx + 1]):
                depth += 1
            else:
                break
        if depth == 0 or depth == n:
            return False
        submatrix = poset_matrix[:-depth, :-depth]
        if not cls.is_connected_poset(submatrix):
            dual_sub = cls.compute_dual_poset_matrix(submatrix)
            if not np.array_equal(dual_sub, submatrix):
                return True
        return False

    @classmethod
    def is_double_dualizable_poset(cls, poset_matrix):
        if poset_matrix.shape[0] <= 2:
            return False
        last_row = poset_matrix[-1]
        first_col = poset_matrix[:, 0]
        if not (np.array_equal(last_row, first_col[::-1]) and
                len(cls.get_maximal_elements(poset_matrix)) == len(cls.get_minimal_elements(poset_matrix))):
            return False
        top_left_sub = poset_matrix[:-1, :-1]
        bottom_right_sub = poset_matrix[1:, 1:]
        if cls.is_semi_right_dualizable(top_left_sub) and cls.is_semi_left_dualizable(bottom_right_sub):
            shared_core_tl = top_left_sub[1:, 1:]
            shared_core_br = bottom_right_sub[:-1, :-1]
            return np.array_equal(shared_core_tl, shared_core_br)
        return False

    @classmethod
    def extract_metrics_bundle(cls, matrix: np.ndarray) -> dict:
        """Computes all structural properties simultaneously for a given matrix object."""
        trivial = cls.is_trivial_poset(matrix)
        connected = cls.is_connected_poset(matrix)
        not_self_dual = cls.is_not_self_dual(matrix)
        semi_right = cls.is_semi_right_dualizable(matrix)
        semi_left = cls.is_semi_left_dualizable(matrix)
        double_dual = cls.is_double_dualizable_poset(matrix)

        return {
            "trivial": trivial,
            "connected": connected,
            "disconnected": not connected if matrix.size > 0 else False,
            "non_self_dual": not_self_dual,
            "self_dual": not not_self_dual,
            "semi_right": semi_right,
            "semi_left": semi_left,
            "double_dual": double_dual
        }

    @classmethod
    def parallel_analyze_dataset(cls, matrices: list[np.ndarray], n_jobs: int = -1) -> list[dict]:
        return Parallel(n_jobs=n_jobs)(delayed(cls.extract_metrics_bundle)(mat) for mat in matrices)


# =============================================================================
# EXECUTABLE BLUEPRINT BENCHMARK WORKBENCH RUNNER
# =============================================================================
if __name__ == "__main__":
    right_side_matrices = np.array([
        [[1, 0], [0, 1]],  # Antichain
        [[1, 0], [1, 1]]   # Chain
    ], dtype='int64')
    left_side_matrices = right_side_matrices

    generation_depth = 8
    n_cpu_cores = -1

    print("======================================================================")
    print("   OPERADIC POSET COMPOSITION: COMBINATORIAL GROWTH WORKBENCH")
    print("======================================================================")
    print(f"Initial Base Element Pool Size : {len(left_side_matrices)}")
    print(f"Active Substitution Pool Size  : {len(right_side_matrices)}")
    print(f"Target Generational Execution Depth: {generation_depth} iterations")
    print("-" * 70)

    experimental_models = {
        "fully_isolated": "Fully Isolated Model  (Theorem 2.1(e))",
        "succ_dominant":  "Successor Dominant Model (Theorem 2.1(d))",
        "pred_dominant":  "Predecessor Dominant Model (Theorem 2.1(c))",
        "isolated":       "Isolated Variant Model (Theorem 2.1(b))",
        "total":          "Total Propagation Model     (Theorem 2.1(a))",
        "general":        "General Composition Model   (Theorem 2.2)",
        "minimal":        "Minimal Variant Model       (Theorem 2.6)",
        "maximal":        "Maximal Variant Model       (Theorem 2.7)",
        "boundary":       "Boundary-Restricted Model   (Theorem 2.8)"
    }

    # Tracking storage matrices map
    generational_growth_records = {}

    for model_key, model_title in experimental_models.items():
        print(f"\n>>> Launching: {model_title}")
        print("." * 60)
        start_time = time.time()

        p_init = left_side_matrices.shape[1]
        q_init = right_side_matrices.shape[1]

        current_signatures = {PosetOperad.matrix_to_bits(m) for m in left_side_matrices}
        insertion_signatures = [PosetOperad.matrix_to_bits(m) for m in right_side_matrices]

        p_current = p_init

        # Initialize dictionary to accumulate historical metrics step counts
        generational_growth_records[model_key] = {
            "total_count": [], "trivial": [], "connected": [], "disconnected": [],
            "non_self_dual": [], "self_dual": [], "semi_right": [], "semi_left": [], "double_dual": []
        }

        for step in range(generation_depth):
            bases_list = list(current_signatures)
            tasks = [
                (l_sig, r_sig, p_current, q_init, model_key)
                for r_sig in insertion_signatures for l_sig in bases_list
            ]
            if not tasks:
                break

            nested_outputs = Parallel(n_jobs=n_cpu_cores, backend="threading")(
                delayed(PosetOperad.generate_compositions_bitpacked)(l, r, p, q, t) for l, r, p, q, t in tasks
            )

            current_signatures = set()
            for sublist in nested_outputs:
                current_signatures.update(sublist)

            p_current = p_current + q_init - 1

            # UNPACK AND REVEAL COMPREHENSIVE GENERATIONAL STAT FILE
            sigs_list = list(current_signatures)
            unpacked_matrices = [PosetOperad.bits_to_matrix(s, p_current) for s in sigs_list]

            # Run concurrent metrics extraction map
            metrics_bundle = PosetProcessor.parallel_analyze_dataset(unpacked_matrices, n_jobs=n_cpu_cores)

            # Reduce collection properties
            generational_growth_records[model_key]["total_count"].append(len(metrics_bundle))
            generational_growth_records[model_key]["trivial"].append(sum(1 for m in metrics_bundle if m["trivial"]))
            generational_growth_records[model_key]["connected"].append(sum(1 for m in metrics_bundle if m["connected"]))
            generational_growth_records[model_key]["disconnected"].append(sum(1 for m in metrics_bundle if m["disconnected"]))
            generational_growth_records[model_key]["non_self_dual"].append(sum(1 for m in metrics_bundle if m["non_self_dual"]))
            generational_growth_records[model_key]["self_dual"].append(sum(1 for m in metrics_bundle if m["self_dual"]))
            generational_growth_records[model_key]["semi_right"].append(sum(1 for m in metrics_bundle if m["semi_right"]))
            generational_growth_records[model_key]["semi_left"].append(sum(1 for m in metrics_bundle if m["semi_left"]))
            generational_growth_records[model_key]["double_dual"].append(sum(1 for m in metrics_bundle if m["double_dual"]))

            print(f" -> [Iteration {step + 1}] Size: {p_current}x{p_current} | Unique Pool: {len(current_signatures)} | Double-Dual: {generational_growth_records[model_key]['double_dual'][-1]}")

        elapsed = time.time() - start_time
        generational_growth_records[model_key]["runtime"] = elapsed
        print(f"Status: Complete | Time Elapsed: {elapsed:.4f} seconds")

    # =============================================================================
    # OUTPUT PRESENTATION MATRIX DISPLAY LOOPS
    # =============================================================================
    display_mapping = [
        ("fully_isolated", "Theorem 2.1(e) (Fully_Isol)"),
        ("succ_dominant",  "Theorem 2.1(d) (Succ_Dominant)"),
        ("pred_dominant",  "Theorem 2.1(c) (Pred_Dominant)"),
        ("isolated",       "Theorem 2.1(b) (Isolated)"),
        ("total",          "Theorem 2.1(a) (Total)"),
        ("general",        "Theorem 2.2    (General)"),
        ("minimal",        "Theorem 2.6    (Minimal)"),
        ("maximal",        "Theorem 2.7    (Maximal)"),
        ("boundary",       "Theorem 2.8    (Boundary)")
    ]

    # SECTION 1: COMPREHENSIVE GENERATIONAL DOUBLE-DUALIZABILITY MATRIX
    print("\n" + "=" * 95)
    print("         GLOBAL INCREMENTAL DOUBLE-DUALIZABILITY STRUCTURAL SEQUENCE MATRIX")
    print("=" * 95)
    header = f"{'Composition Model Strategy':<30} | " + " | ".join([f"Gen {i}" for i in range(1, 9)]) + " | Runtime"
    print(header)
    print("-" * 125)
    for model_key, label in display_mapping:
        data = generational_growth_records[model_key]
        hist = data["double_dual"]
        cols = " | ".join([f"{val:<6}" for val in hist])
        print(f"{label:<30} | {cols} | {data['runtime']:.3f}s")
    print("=" * 125)

    # SECTION 2: BREAKDOWN BY CRITERIA FOR ALL GENERATIONAL PATHS
    for model_key, label in display_mapping:
        print("\n" + "=" * 95)
        print(f" STRUCTURAL STATISTICS HISTOGRAM SEQUENCE: {label.upper()}")
        print("=" * 95)
        print(f"{'Order Property Criterion':<30} | " + " | ".join([f"Gen {i}" for i in range(1, 9)]))
        print("-" * 115)
        data = generational_growth_records[model_key]

        for property_key in ["total_count", "trivial", "connected", "disconnected", "non_self_dual", "self_dual", "semi_right", "semi_left", "double_dual"]:
            hist = data[property_key]
            cols = " | ".join([f"{val:<6}" for val in hist])
            print(f"{property_key.replace('_', ' ').title():<30} | {cols}")
        print("=" * 95)


   OPERADIC POSET COMPOSITION: COMBINATORIAL GROWTH WORKBENCH
Initial Base Element Pool Size : 2
Active Substitution Pool Size  : 2
Target Generational Execution Depth: 8 iterations
----------------------------------------------------------------------

>>> Launching: Fully Isolated Model  (Theorem 2.1(e))
............................................................
 -> [Iteration 1] Size: 3x3 | Unique Pool: 3 | Double-Dual: 0
 -> [Iteration 2] Size: 4x4 | Unique Pool: 5 | Double-Dual: 0
 -> [Iteration 3] Size: 5x5 | Unique Pool: 8 | Double-Dual: 0
 -> [Iteration 4] Size: 6x6 | Unique Pool: 13 | Double-Dual: 0
 -> [Iteration 5] Size: 7x7 | Unique Pool: 21 | Double-Dual: 0
 -> [Iteration 6] Size: 8x8 | Unique Pool: 34 | Double-Dual: 0
 -> [Iteration 7] Size: 9x9 | Unique Pool: 55 | Double-Dual: 0
 -> [Iteration 8] Size: 10x10 | Unique Pool: 89 | Double-Dual: 0
Status: Complete | Time Elapsed: 3.5487 seconds

>>> Launching: Successor Dominant Model (Theorem 2.1(d))
......................